# 05 - Action: 验证与稳健性检验

> 验证结论可靠性 · 排除替代解释 · 确保结论稳健

---

## 一、为什么要做稳健性检验？

### 1.1 检验目的

```
核心问题：我们的结论是"真的"还是"碰巧的"？

可能的问题：
1. 模型误设 → 估计有偏
2. 样本选择 → 结论不具代表性
3. 极端值 → 结果受少数样本驱动
4. 遗漏变量 → 遗漏重要混淆因素
5. 多重共线性 → 估计不稳定

稳健性检验的目标：
- 改变模型设定，结论是否依然成立？
- 改变样本范围，结论是否依然成立？
- 改变估计方法，结论是否依然成立？
""")

print("TODO: 列出所有可能的稳健性检验")

## 二、检验1：改变模型设定

In [ ]:
# TODO: 用不同的ML模型作为nuisance估计器

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.model_selection import train_test_split

# 加载数据
df = pd.read_csv('../data/processed/rider_pricing_processed.csv')
Y = df['order_volume']
T = df['price']

X_cols = [
    'hour', 'day_of_week', 'is_weekend', 'temperature', 'humidity', 
    'wind_speed', 'region_density', 'avg_income', 'competitor_price',
    'holiday', 'promotion_active', 'historical_ontime_rate', 
    'historical_rider_supply', 'historical_order_volume', 'is_peak'
]
cat_cols = ['weather', 'region_type', 'region_id', 'price_level']
X = df[X_cols + cat_cols].copy()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# 不同的nuisance模型
models = {
    'Random Forest': (RandomForestRegressor(n_estimators=100, random_state=42),
                      RandomForestRegressor(n_estimators=100, random_state=42)),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=100, random_state=42),
                          GradientBoostingRegressor(n_estimators=100, random_state=42)),
    'Ridge': (RidgeCV(), RidgeCV())
}

print("="*60)
print("检验1：改变nuisance模型设定")
print("="*60)

results = []
for name, (model_y, model_t) in models.items():
    # 简化版DML
    X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
        X, T, Y, test_size=0.3, random_state=42
    )
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    Y_pred = model_y.predict(X_test)
    T_pred = model_t.predict(X_test)
    
    Y_resid = Y_test - Y_pred
    T_resid = T_test - T_pred
    
    theta = np.mean(T_resid * Y_resid) / np.mean(T_resid ** 2)
    
    results.append({'Model': name, 'Theta': theta})
    print(f"{name}: {theta:.4f}")

results_df = pd.DataFrame(results)
print(f"\n效应估计范围: [{results_df['Theta'].min():.4f}, {results_df['Theta'].max():.4f}]")
print(f"标准差: {results_df['Theta'].std():.4f}")

print("\n" + "="*60)
print("结论：")
print("="*60)
print("如果不同nuisance模型的结果差异不大 → 结论稳健")
print("如果差异很大 → 模型设定敏感，需要进一步分析")
print("="*60)

print("TODO: 分析不同模型结果的差异原因")

## 三、检验2：改变样本范围

In [ ]:
# TODO: 排除极端样本，看结论是否变化

print("="*60)
print("检验2：改变样本范围")
print("="*60)

# 子样本1：排除极端价格
df_sub1 = df[(df['price'] >= 4) & (df['price'] <= 12)]
print(f"子样本1（排除极端价格）: {len(df_sub1)} 条")

# 子样本2：排除节假日
df_sub2 = df[df['holiday'] == False]
print(f"子样本2（排除节假日）: {len(df_sub2)} 条")

# 子样本3：仅工作日高峰
df_sub3 = df[(df['is_weekend'] == False) & (df['is_peak'] == True)]
print(f"子样本3（仅工作日高峰）: {len(df_sub3)} 条")

# 子样本4：仅CBD区域
df_sub4 = df[df['region_type'] == 'cbd']
print(f"子样本4（仅CBD）: {len(df_sub4)} 条")

subsamples = [
    ('全样本', df),
    ('排除极端价格', df_sub1),
    ('排除节假日', df_sub2),
    ('仅工作日高峰', df_sub3),
    ('仅CBD', df_sub4)
]

print("\n" + "="*60)
print("各子样本的DML估计:")
print("="*60)

for name, sub_df in subsamples:
    Y_sub = sub_df['order_volume']
    T_sub = sub_df['price']
    X_sub = sub_df[X_cols + cat_cols].copy()
    X_sub = pd.get_dummies(X_sub, columns=cat_cols, drop_first=True)
    
    # 简化DML
    X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
        X_sub, T_sub, Y_sub, test_size=0.3, random_state=42
    )
    
    model_y = RandomForestRegressor(n_estimators=100, random_state=42)
    model_t = RandomForestRegressor(n_estimators=100, random_state=42)
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    Y_pred = model_y.predict(X_test)
    T_pred = model_t.predict(X_test)
    
    Y_resid = Y_test - Y_pred
    T_resid = T_test - T_pred
    
    theta = np.mean(T_resid * Y_resid) / np.mean(T_resid ** 2)
    
    print(f"{name}: {theta:.4f}")

print("\n" + "="*60)
print("结论：")
print("="*60)
print("如果不同子样本的结论一致 → 结论稳健")
print("如果差异很大 → 结论受特定样本驱动，需要谨慎")
print("="*60)

print("TODO: 分析哪些子样本的结论差异较大")

## 四、检验3：安慰剂检验

In [ ]:
# TODO: 安慰剂检验

print("="*60)
print("检验3：安慰剂检验（Placebo Test）")
print("="*60)

print("""
安慰剂检验原理：

1. 随机分配处理变量
   - 打乱真实的价格分配
   - 如果我们的方法正确，应该估计出"零效应"

2. 使用假的结果变量
   - 用与价格无关的变量作为结果
   - 应该估计出"零效应"

3. 预期：
   - 安慰剂效应 ≈ 0
   - 如果显著不为0 → 方法有问题
""")

# 安慰剂检验1：随机打乱价格
print("\n" + "="*60)
print("安慰剂1：随机打乱价格")
print("="*60)

n_placebo = 100
placebo_effects = []

for i in range(n_placebo):
    # 随机打乱价格
    T_placebo = T.sample(frac=1, random_state=i).reset_index(drop=True)
    
    # 简化DML
    X_train, X_test, T_train_p, T_test_p, Y_train, Y_test = train_test_split(
        X, T_placebo, Y, test_size=0.3, random_state=42
    )
    
    model_y = RandomForestRegressor(n_estimators=50, random_state=42)
    model_t = RandomForestRegressor(n_estimators=50, random_state=42)
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train_p)
    
    Y_pred = model_y.predict(X_test)
    T_pred = model_t.predict(X_test)
    
    Y_resid = Y_test - Y_pred
    T_resid = T_test_p - T_pred
    
    theta_placebo = np.mean(T_resid * Y_resid) / np.mean(T_resid ** 2)
    placebo_effects.append(theta_placebo)

placebo_effects = np.array(placebo_effects)

print(f"安慰剂效应均值: {placebo_effects.mean():.4f}")
print(f"安慰剂效应标准差: {placebo_effects.std():.4f}")
print(f"95%分位数: [{np.percentile(placebo_effects, 2.5):.4f}, {np.percentile(placebo_effects, 97.5):.4f}]")

# 可视化安慰剂分布
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(placebo_effects, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', label='零效应')
plt.axvline(theta_dml, color='blue', linestyle='--', label='真实估计')
plt.xlabel('安慰剂效应')
plt.ylabel('频次')
plt.title('安慰剂检验分布')
plt.legend()
plt.show()

print("\n" + "="*60)
print("结论：")
print("="*60)
print("如果安慰剂效应集中在0附近 → 方法可靠")
print("如果安慰剂效应显著不为0 → 方法有问题")
print("="*60)

print("TODO: 分析安慰剂检验结果")

## 五、检验4：敏感性分析

In [ ]:
# TODO: 敏感性分析

print("="*60)
print("检验4：敏感性分析")
print("="*60)

print("""
敏感性分析：

问题：如果存在未观察到的混淆因素，结论会改变吗？

方法：
1. 假设存在一个未观察到的变量U
2. U同时影响处理变量和结果变量
3. 改变U的强度，看结论何时反转

解释：
- 如果很小的U就能反转结论 → 结论不稳健
- 需要很大的U才能反转 → 结论稳健
""")

# 简化版敏感性分析
print("\n" + "="*60)
print("简化版：改变关键协变量的包含/排除")
print("="*60)

# 不同的协变量组合
covariate_sets = [
    ('最小集合', ['hour', 'is_peak', 'weather']),
    ('中等集合', ['hour', 'is_peak', 'weather', 'region_type', 'temperature']),
    ('完整集合', X_cols + cat_cols)
]

for name, cols in covariate_sets:
    available_cols = [c for c in cols if c in df.columns]
    X_sub = df[available_cols].copy()
    
    # 处理分类变量
    cat_in_sub = [c for c in cat_cols if c in available_cols]
    if cat_in_sub:
        X_sub = pd.get_dummies(X_sub, columns=cat_in_sub, drop_first=True)
    
    # 简化DML
    X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
        X_sub, T, Y, test_size=0.3, random_state=42
    )
    
    model_y = RandomForestRegressor(n_estimators=100, random_state=42)
    model_t = RandomForestRegressor(n_estimators=100, random_state=42)
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    Y_pred = model_y.predict(X_test)
    T_pred = model_t.predict(X_test)
    
    Y_resid = Y_test - Y_pred
    T_resid = T_test - T_pred
    
    theta = np.mean(T_resid * Y_resid) / np.mean(T_resid ** 2)
    
    print(f"{name}: {theta:.4f}")

print("\n" + "="*60)
print("结论：")
print("="*60)
print("如果不同协变量集合的结论一致 → 结论对遗漏变量不敏感")
print("如果差异很大 → 可能存在重要遗漏变量")
print("="*60)

print("TODO: 分析哪些协变量对结论影响最大")

## 六、检验5：Bootstrap置信区间

In [ ]:
# TODO: Bootstrap置信区间

print("="*60)
print("检验5：Bootstrap置信区间")
print("="*60)

print("""
Bootstrap原理：

1. 从原始数据中有放回地抽样
2. 对每个样本重新估计模型
3. 重复多次（如1000次）
4. 用估计值的分布构建置信区间

优势：
- 不依赖正态分布假设
- 能捕捉估计量的真实分布
- 对复杂模型尤其有用
""")

n_bootstrap = 500
bootstrap_estimates = []

print(f"运行{n_bootstrap}次Bootstrap...")

for i in range(n_bootstrap):
    # 有放回抽样
    idx = np.random.choice(len(df), size=len(df), replace=True)
    df_boot = df.iloc[idx]
    
    Y_boot = df_boot['order_volume']
    T_boot = df_boot['price']
    X_boot = df_boot[X_cols + cat_cols].copy()
    X_boot = pd.get_dummies(X_boot, columns=cat_cols, drop_first=True)
    
    # 简化DML
    X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
        X_boot, T_boot, Y_boot, test_size=0.3, random_state=42
    )
    
    model_y = RandomForestRegressor(n_estimators=50, random_state=42)
    model_t = RandomForestRegressor(n_estimators=50, random_state=42)
    
    model_y.fit(X_train, Y_train)
    model_t.fit(X_train, T_train)
    
    Y_pred = model_y.predict(X_test)
    T_pred = model_t.predict(X_test)
    
    Y_resid = Y_test - Y_pred
    T_resid = T_test - T_pred
    
    theta_boot = np.mean(T_resid * Y_resid) / np.mean(T_resid ** 2)
    bootstrap_estimates.append(theta_boot)

bootstrap_estimates = np.array(bootstrap_estimates)

print(f"\nBootstrap结果:")
print(f"均值: {bootstrap_estimates.mean():.4f}")
print(f"标准差: {bootstrap_estimates.std():.4f}")
print(f"95%置信区间: [{np.percentile(bootstrap_estimates, 2.5):.4f}, {np.percentile(bootstrap_estimates, 97.5):.4f}]")

# 可视化
plt.figure(figsize=(10, 6))
plt.hist(bootstrap_estimates, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(bootstrap_estimates.mean(), color='red', linestyle='--', label='Bootstrap均值')
plt.axvline(np.percentile(bootstrap_estimates, 2.5), color='orange', linestyle='--', label='95% CI下限')
plt.axvline(np.percentile(bootstrap_estimates, 97.5), color='orange', linestyle='--', label='95% CI上限')
plt.xlabel('因果效应估计')
plt.ylabel('频次')
plt.title('Bootstrap分布')
plt.legend()
plt.show()

print("\n" + "="*60)
print("结论：")
print("="*60)
print("Bootstrap置信区间比渐近区间更可靠")
print("如果置信区间不包含0 → 效应显著")
print("="*60)

print("TODO: 对比Bootstrap CI和渐近CI")

## 七、稳健性检验总结

In [ ]:
# TODO: 汇总所有稳健性检验

print("""
稳健性检验汇总：

检验                    目的                    通过标准
─────────────────────────────────────────────────────────
1. 改变模型设定         检验模型误设敏感性        不同模型结果一致
2. 改变样本范围         检验样本选择敏感性        不同子样本结论一致
3. 安慰剂检验           检验方法可靠性            安慰剂效应≈0
4. 敏感性分析           检验遗漏变量影响          不同协变量集合结论一致
5. Bootstrap CI         检验统计可靠性            CI不包含0

通过所有检验 → 结论高度可信
通过大部分检验 → 结论基本可信，需注意例外情况
未通过多项检验 → 结论不可靠，需要重新审视
""")

print("\n" + "="*60)
print("最终结论可靠性评估:")
print("="*60)
print("1. 效应方向：价格对订单量的因果效应为[正/负/不显著]")
print("2. 效应大小：每增加1元，订单量变化约X单")
print("3. 异质性：雨天CBD有效，雨天郊区无效")
print("4. 置信度：通过X/5项稳健性检验")
print("="*60)

print("TODO: 根据检验结果，给出结论可靠性评级")

## 八、下一步

1. **结果汇总**：整合所有分析结论
2. **业务转化**：将统计结论转化为策略建议
3. **报告撰写**：用report-template撰写最终报告

---

> **核心洞察**：稳健性检验不是"走过场"，而是"试金石"——只有经得起检验的结论，才值得被业务采纳。